In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.float_format', '{:.6f}'.format)
%matplotlib inline


In [26]:
IN_PATH = 'CLEANED DATA/REMX_prices_sentiment_combined_without_weekends.xlsx'

df = pd.read_excel(IN_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')
df['r'] = np.log(df['Close'] / df['Close'].shift(1))
r  = df['r'].dropna()
r2 = (r ** 2)


## Step 6 — Sentiment-augmented EGARCH(1,1)-X by MLE

We now augment the baseline GARCH(1,1) by (i) replacing the symmetric ARCH term with an EGARCH log-variance specification that allows a sign asymmetry through the leverage parameter $\xi$, and (ii) injecting two exogenous sentiment regressors into the variance equation:

$$\ln(\sigma_t^2) \;=\; \omega \;+\; \alpha\bigl[\,|z_{t-1}| - \mathbb{E}|z_{t-1}|\bigr] \;+\; \xi\, z_{t-1} \;+\; \beta\, \ln(\sigma_{t-1}^2) \;+\; \gamma_1\, \text{Tone}_{m,t-1} \;+\; \gamma_2\, \text{Volume}_{m,t-1},$$

where $z_t = \varepsilon_t / \sigma_t$ is the standardized residual, with $\mathbb{E}|z_t| = \sqrt{2/\pi}$ under the Gaussian assumption. The parameters $(\mu, \omega, \alpha, \xi, \beta, \gamma_1, \gamma_2)$ are estimated by Maximum Likelihood (Gaussian innovations) with robust sandwich standard errors $H^{-1} J H^{-1}$. The results are reported in the same format as the GARCH(1,1) baseline so that log-likelihood, AIC and BIC are directly comparable.

In [27]:
%%capture cap_egarch_x_normal
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

# ── 1. Aligned dataset: r_t (in %), Tone_{t-1}, log_article_count_{t-1} ──────────
r_pct    = df['r'] * 100.0
tone_lag = df['tone_mean'].shift(1)
vol_lag  = df['art_growth'].shift(1)

aligned = pd.concat(
    [r_pct, tone_lag, vol_lag],
    axis=1, keys=['r', 'tone_lag', 'vol_lag']
).dropna()

r    = aligned['r'].values
tone = aligned['tone_lag'].values
vol  = aligned['vol_lag'].values
T    = len(r)

print(f'Sample after alignment: T = {T} obs '
      f'({aligned.index.min().date()} -> {aligned.index.max().date()})')

# ── 2. EGARCH(1,1)-X recursion (returns per-obs log-likelihood) ───────────
SQRT_2_OVER_PI = np.sqrt(2.0 / np.pi)
PARAM_NAMES = ['mu', 'omega', 'alpha', 'xi', 'beta', 'gamma_tone', 'gamma_vol']
CLAMP = 30.0  # numerical guard on log_sig2 during optimization

def per_obs_ll(params, r, tone, vol):
    mu, omega, alpha, xi, beta, g1, g2 = params
    n = len(r)
    eps = r - mu
    log_sig2 = np.empty(n)
    log_sig2[0] = np.log(max(np.var(eps), 1e-8))

    for t in range(1, n):
        z_prev = eps[t-1] / np.exp(0.5 * log_sig2[t-1])
        ls = (omega
              + alpha * (np.abs(z_prev) - SQRT_2_OVER_PI)
              + xi    * z_prev
              + beta  * log_sig2[t-1]
              + g1    * tone[t]
              + g2    * vol[t])
        log_sig2[t] = np.clip(ls, -CLAMP, CLAMP)

    ll_t = -0.5 * (np.log(2.0 * np.pi) + log_sig2 + eps**2 / np.exp(log_sig2))
    return ll_t, log_sig2

def neg_ll(params, r, tone, vol):
    ll_t, _ = per_obs_ll(params, r, tone, vol)
    return -ll_t.sum()

# ── 3. Maximum-likelihood estimation ──────────────────────────────────────
x0 = np.array([0.0, -0.10, 0.10, -0.05, 0.95, 0.0, 0.0])
opt = minimize(neg_ll, x0, args=(r, tone, vol),
               method='L-BFGS-B', options={'disp': False, 'maxiter': 5000})
if not opt.success:
    print(f'WARNING: optimizer message: {opt.message}')

theta = opt.x
ll    = -opt.fun

# ── 4. Robust (sandwich) standard errors ──────────────────────────────────
H      = approx_hess(theta, neg_ll, args=(r, tone, vol))
V_hess = np.linalg.inv(H)

# Numerical score matrix (per-observation gradients), central differences
eps_fd = 1e-5
G = np.empty((T, len(theta)))
for i in range(len(theta)):
    p_up = theta.copy(); p_up[i] += eps_fd
    p_dn = theta.copy(); p_dn[i] -= eps_fd
    ll_up, _ = per_obs_ll(p_up, r, tone, vol)
    ll_dn, _ = per_obs_ll(p_dn, r, tone, vol)
    G[:, i] = (ll_up - ll_dn) / (2 * eps_fd)

J        = G.T @ G                # outer product of scores (BHHH)
V_robust = V_hess @ J @ V_hess    # sandwich
se       = np.sqrt(np.diag(V_robust))
t_ratios = theta / se
p_values = 2.0 * (1.0 - norm.cdf(np.abs(t_ratios)))

# ── 5. Information criteria ───────────────────────────────────────────────
k   = len(theta)
aic = -2.0 * ll + 2.0 * k
bic = -2.0 * ll + k * np.log(T)

# ── 6. Report  ─────────────────────────────────
tbl = pd.DataFrame({
    'estimate': theta,
    'std_err':  se,
    't_ratio':  t_ratios,
    'p_value':  p_values,
}, index=PARAM_NAMES)
tbl['significant_10pct'] = tbl['t_ratio'].abs() > 1.645
tbl['significant_5pct']  = tbl['t_ratio'].abs() > 1.96
tbl['significant_1pct']  = tbl['t_ratio'].abs() > 2.576

print()
print('Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):')
print(tbl.to_string(float_format=lambda x: f'{x: .4f}'))
print()
print(f'Log-likelihood : {ll: .4f}')
print(f'AIC            : {aic: .4f}')
print(f'BIC            : {bic: .4f}')
print()
beta_hat = theta[4]
print(f'beta = {beta_hat:.4f}   '
      f'({"stationary (|beta|<1)" if abs(beta_hat) < 1 else "non-stationary (|beta|>=1)"})')
half_life = np.log(0.5) / np.log(abs(beta_hat)) if 0 < abs(beta_hat) < 1 else float('inf')
print(f'Half-life of a log-variance shock: {half_life:.1f} trading days')
print()
print('Verdict:')
for name, row in tbl.iterrows():
    if row['significant_1pct']:
        tag = 'SIGNIFICANT at 1% (and 5%, 10%)'
    elif row['significant_5pct']:
        tag = 'SIGNIFICANT at 5% (and 10%) only'
    elif row['significant_10pct']:
        tag = 'SIGNIFICANT at 10% only'
    else:
        tag = 'not significant'
    print(f'  {name:<12}  t = {row["t_ratio"]:+.2f}   p = {row["p_value"]:.4f}   -> {tag}')

In [28]:
import json
from pathlib import Path

REPORTS_DIR = Path("REPORTS")
REPORTS_DIR.mkdir(exist_ok=True)

# Persist capture and the Gaussian EGARCH-X log-likelihood for notebook 05
(REPORTS_DIR / "_capture_egarch_x_normal.txt").write_text(cap_egarch_x_normal.stdout)
with (REPORTS_DIR / "egarch_x_normal_ll.json").open("w") as fh:
    json.dump({'ll_gauss_egarch_x': float(ll), 'T': int(T)}, fh)

print(cap_egarch_x_normal.stdout)
print(f"Persisted capture -> REPORTS/_capture_egarch_x_normal.txt")
print(f"Persisted Gaussian EGARCH-X log-likelihood ({ll:.4f}) -> REPORTS/egarch_x_normal_ll.json")


Sample after alignment: T = 2765 obs (2015-04-02 -> 2026-03-31)

Per-parameter significance (|t| > 1.645 -> 10%, |t| > 1.96 -> 5%, |t| > 2.576 -> 1%):
            estimate  std_err   t_ratio  p_value  significant_10pct  significant_5pct  significant_1pct
mu           -0.0207   0.0369   -0.5606   0.5750              False             False             False
omega         0.0389   0.0117    3.3257   0.0009               True              True              True
alpha         0.1697   0.0279    6.0831   0.0000               True              True              True
xi           -0.0230   0.0135   -1.7073   0.0878               True             False             False
beta          0.9783   0.0073  134.9002   0.0000               True              True              True
gamma_tone    0.0060   0.0060    0.9869   0.3237              False             False             False
gamma_vol    -0.0175   0.0085   -2.0433   0.0410               True              True             False

Log-likelihood :